In [1]:
!pip install mne autoreject eegprep

  Using cached eegprep-0.2.23-py3-none-any.whl.metadata (9.5 kB)
  Using cached neo-0.14.3-py3-none-any.whl.metadata (9.2 kB)
  Using cached oct2py-5.8.0-py3-none-any.whl.metadata (5.0 kB)
  Using cached pybids-0.21.0-py3-none-any.whl.metadata (8.1 kB)
  Using cached pyedflib-0.1.42.tar.gz (2.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached quantities-0.16.4-py3-none-any.whl.metadata (4.1 kB)
  Using cached octave_kernel-0.36.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached metakernel-0.30.4-py3-none-any.whl.metadata (7.4 kB)
  Using cached formulaic-1.2.1-py3-none-any.whl.metadata (7.0 kB)
  Using cached bids_validator-1.14.

  error: subprocess-exited-with-error
  
  × Building wheel for pyedflib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [46 lines of output]
      C:\Users\tramy\AppData\Local\Temp\pip-build-env-3shkcb4d\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: BSD License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-313

In [ ]:
def preprocessing(path,
                  sub,
                  ses,
                  task='meditation',
                  resampling_frq=None,
                  ref_chs=None,
                  filter_bounds=None,
                  ica_n_components=None,
                  autoreject='global',
                  ica_report=True,
                  n_job=1):


    pos = mne.channels.make_standard_montage('biosemi64')

    # open bids
    raw = mne.io.read_raw_bdf(path, extra_params={'preload': True}, verbose=False)

    raw.set_channel_types({'EXG1': 'eog', 'EXG2': 'eog', 'EXG3': 'eog', 'EXG4': 'eog',  # This needs to be updated
                'EXG5': 'emg', 'EXG6': 'emg', 'EXG7': 'emg', 'EXG8': 'emg'})

    # set channels positions
    raw.set_montage(pos)

    # interpolate bad channels
    if raw.info['bads'] != []:
        raw.interpolate_bads()

    # resampling
    if resampling_frq is not None:  # you don't need to resample
        raw.resample(resampling_frq)

    # filtering
    if filter_bounds is not None:
        raw.filter(
            l_freq=filter_bounds[0], # 1 Hz
            h_freq=filter_bounds[1], # 42 Hz
            n_jobs=n_job
            )

    # apply ICA, remove eog and ecg ICs using templates, and save the report
    raw = run_ica(raw,
                  sub,
                  task,
                  ses,
                  ica_path_root='data/ds001787-download/ica', #TODO you need to update this
                  ica_report_root='data/ds001787-download/ica_reports', #TODO you need to update this
                  n_components=ica_n_components,
                  report=ica_report)

    # epoching (note: for creating epochs with mne.epochs, tmin and tmax should be specified!)
    epochs = mne.make_fixed_length_epochs(raw, duration=1, preload=True)
    del raw

    # autoreject
    if autoreject == 'global':
        # pick only eeg channels for getting rejection threshold:
        reject = get_rejection_threshold(epochs.copy().pick_types(eeg=True))
        epochs.drop_bad(reject=reject)

    if autoreject == 'local':
        ar = AutoReject()
        epochs = ar.fit_transform(epochs)

    # rereferencing
    if ref_chs is not None:
        epochs.add_reference_channels(ref_channels='FCz')  # adding reference channel to the data
        epochs.set_eeg_reference(ref_channels=ref_chs)

    # save clean epochs
    epochs.save(f'data/ds001787-download/clean_data/sub-{sub}_ses-{ses}_task-{task}_proc-clean_epo.fif', overwrite=True) #TODO you need to update this

def run_ica(raw,
        sub,
        task,
        ses,
        ica_path_root,
        filter_beforeICA=False,
        n_components=None,
        random_state=97,
        show_plot=False,
        threshold=0.85,
        report=True):

    ica_path = ica_path_root + f'/sub-{sub}_ses-{ses}_task-{task}_ica.fif'

    # open templates
    eog_ica, eog_3rd_ica, eog_inds = _open_templates()

    # ica
    ica = ICA(n_components, random_state=random_state)
    if Path(ica_path).exists():
        ica = read_ica(ica_path)

    else:
        raw_filt = raw.copy().filter(1, None) if filter_beforeICA else raw
        ica.fit(raw_filt)

    # create list of icas
    icas_eog = [eog_ica]
    icas_eog.append(ica)

    # detect eog components
    eog_epochs = mne.preprocessing.create_eog_epochs(raw=raw)  # noqa
    _, eog_scores = ica.find_bads_eog(raw, ch_name=['EOG1', 'EOG2', 'Fpz'])  #TODO: you need to replace the EOG1 and EOG2 names based on the channel names in the new dataset
    # we know that there is two components in the eog template so:
    [corrmap(icas_eog, template=(0, eog_inds[i]), threshold=threshold, label='blink', plot=show_plot)
     for i in range(2)]
    # detecting oculomotor activity using third compenents
    icas_eog.append(eog_3rd_ica)
    eog_inds.append(7)  # use components 7 from 3rd_ica as a template:
    corrmap(icas_eog, template=(2, eog_inds[2]), threshold=threshold, label='oculomotor', plot=show_plot)

    if 'blink' in ica.labels_.keys():
        eog_comps = ica.labels_['blink']
    else:
        eog_comps = []

    ica.exclude = eog_comps
    print(f'EOG COMPONENTS: {ica.labels_}')

    if not Path(ica_path).exists():
        ica.save(ica_path)

    # apply ica
    print(f'Applying ICA for sub-{sub}...')
    ica.apply(raw, exclude=ica.exclude)

    return raw


def _open_templates(fname_eog_ica='data/ica/templates/eog-ica.fif', #TODO you need to update this
                    fname_eog_3rd_ica='data/ica/templates/eog_3rd-ica.fif',
                    fname_eog_template='data/ica/templates/eog_template.npy',
                    ):
    # read ica and template
    eog_ica = read_ica(fname_eog_ica)
    eog_3rd_ica = read_ica(fname_eog_3rd_ica)
    eog_inds = np.load(fname_eog_template, allow_pickle=True)
    eog_inds = eog_inds[0]

    return eog_ica, eog_3rd_ica, eog_inds

In [ ]:
# USAGE EXAMPLE: This example open and preprocess one eeg data, before running this cell make sure the file addresses are correct

subject = '001'
session = '01'

eeg_path = '/content/sub-001_ses-01_task-meditation_eeg.bdf' # example, should be updated

preprocessing(eeg_path,
              subject,
              session,
              ref_chs='average',
              filter_bounds=[1, 42])